# Lab 7B — Canaries, Safe Rollback, and Agent Bricks

[Lab 6B](#) registered a tuned agent. This lab asks the operational question:
**how do you ship a change to it without finding out from a customer?**

| Cell group | What you do |
|---|---|
| 1–2 | Register a candidate with one line of its prompt loosened. Alias both. |
| 3 | Canary them on grounded and ungrounded probes. |
| 4 | **Repeat the probe. The regression is intermittent.** |
| 5 | Promote, then roll back, by moving a pointer. |

Step 5 of the **guide** covers Agent Bricks, which is console-only.

**Compute:** Serverless. **Prerequisite:** [Lab 6B](#) — a `READY` model version.

In [0]:
%pip install -q "mlflow[databricks]" databricks-agents databricks-ai-search openai
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 1. A candidate, with one line changed

This is the change that gets requested after someone reads a week of refusals in
the logs and asks the agent to stop saying *"I don't know"*.

```diff
-SYSTEM = ("You are a support agent … Answer only from the policy excerpts you
-          retrieve. If they do not cover the question, say so. …")
+SYSTEM = ("You are a helpful support agent … Use the policy excerpts you retrieve.
+          Always give the customer a useful, confident answer. …")
```

Retrieval is untouched: same index, same `k=2`, same `audience: customer` filter.

In [0]:
%%writefile serving_agent_v3.py
"""Candidate v3 — identical retrieval, deliberately loosened instruction."""
import os, json
import mlflow
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse

INDEX = "agents_labs.retail.support_chunks_idx"
ENDPOINT = "agents-labs-vs"
CHAT = "databricks-claude-sonnet-5"
RETRIEVAL_K = 2

SYSTEM = ("You are a helpful support agent for an office furniture retailer. "
          "Use the policy excerpts you retrieve. Always give the customer a useful, "
          "confident answer. Cite a document id like [DOC-003] where you can.")


class SupportAgent(ResponsesAgent):
    def _clients(self):
        from databricks.sdk import WorkspaceClient
        from openai import OpenAI
        from databricks.ai_search.client import VectorSearchClient
        w = WorkspaceClient()
        tok = w.config.authenticate()["Authorization"].removeprefix("Bearer ")
        oa = OpenAI(api_key=tok, base_url=f"{w.config.host}/serving-endpoints")
        vsc = VectorSearchClient(workspace_url=w.config.host,
                                 personal_access_token=tok, disable_notice=True)
        return oa, vsc.get_index(endpoint_name=ENDPOINT, index_name=INDEX)

    @mlflow.trace(span_type="RETRIEVER")
    def _search(self, ix, query):
        r = ix.similarity_search(query_text=query,
                                 columns=["chunk_id", "doc_id", "title", "chunk"],
                                 filters={"audience": "customer"},
                                 num_results=RETRIEVAL_K)
        return [{"doc_id": x[1], "title": x[2], "text": x[3]}
                for x in (r.get("result", {}).get("data_array", []) or [])]

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        oa, ix = self._clients()
        q = ""
        for m in request.input:
            m = m if isinstance(m, dict) else m.__dict__
            if m.get("role") == "user":
                q = m.get("content") or ""
        hits = self._search(ix, q)
        ctx = "\n\n".join(f"[{h['doc_id']}] {h['text']}" for h in hits)
        msg = oa.chat.completions.create(
            model=CHAT, max_tokens=700,
            messages=[{"role": "system", "content": SYSTEM},
                      {"role": "user", "content": f"Excerpts:\n{ctx}\n\nQuestion: {q}"}],
        ).choices[0].message
        c = msg.content
        text = c if isinstance(c, str) else "\n".join(
            b.get("text", "") for b in (c or [])
            if isinstance(b, dict) and b.get("type") == "text")
        return ResponsesAgentResponse(output=[{
            "type": "message", "role": "assistant", "id": "sa-1",
            "content": [{"type": "output_text", "text": text}]}])


mlflow.models.set_model(SupportAgent())

Overwriting serving_agent_v3.py


## 2. Register it, and name the versions

A version number is not a rollout. **Aliases** are: `@champion` is whatever serves
traffic, `@challenger` is whatever you are testing. Callers reference the alias,
never the number, so promotion and rollback become a pointer move.

In [0]:
import os, time, re, json, warnings
import mlflow
from mlflow import MlflowClient
from databricks.sdk import WorkspaceClient
from mlflow.models.resources import (DatabricksServingEndpoint,
                                     DatabricksVectorSearchIndex, DatabricksFunction)

warnings.filterwarnings("ignore", message=".*serializer warnings.*")
CATALOG, SCHEMA = "agents_labs", "retail"
UC_MODEL = f"{CATALOG}.{SCHEMA}.support_agent"

w = WorkspaceClient()
user = w.current_user.me().user_name
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(f"/Users/{user}/agents-labs-7b")

with mlflow.start_run(run_name="candidate-loosened-prompt"):
    mlflow.log_param("variant", "loosened-prompt")
    info = mlflow.pyfunc.log_model(
        name="agent", python_model="serving_agent_v3.py",
        registered_model_name=UC_MODEL,
        resources=[
            DatabricksServingEndpoint(endpoint_name="databricks-claude-sonnet-5"),
            DatabricksServingEndpoint(endpoint_name="databricks-gte-large-en"),
            DatabricksVectorSearchIndex(index_name=f"{CATALOG}.{SCHEMA}.support_chunks_idx"),
            DatabricksFunction(function_name=f"{CATALOG}.{SCHEMA}.get_order_summary"),
        ],
        pip_requirements=["mlflow>=3.11", "databricks-sdk",
                          "databricks-ai-search", "openai"])

new = info.registered_model_version
c = MlflowClient()

# Pick the incumbent from READY versions only. Failed registrations linger in the
# version list forever, and aliasing @champion to one is unrecoverable at 2am.
ready = sorted((int(m.version) for m in c.search_model_versions(f"name='{UC_MODEL}'")
                if m.status == "READY"), reverse=True)
incumbent = next((v for v in ready if v != int(new)), None)

c.set_registered_model_alias(UC_MODEL, "champion", str(incumbent))
c.set_registered_model_alias(UC_MODEL, "challenger", str(new))

print(f"  registered v{new}")
print(f"  READY versions: {ready}")
print(f"\n  @champion   -> v{incumbent}")
print(f"  @challenger -> v{new}")
print(f"\n  callers load models:/{UC_MODEL}@champion and never a version number")

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc
🔗 View Logged Model at: https://adb-7405616584960877.17.azuredatabricks.net/ml/experiments/1670926665559538/models/m-b43025ba0b2e4d8eac1661b1a165852c?o=7405616584960877
2026/09/30 12:06:48 INFO mlflow.pyfunc: Predicting on input example to validate output


[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Registered model 'agents_labs.retail.support_agent' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '9' of model 'agents_labs.retail.support_agent': https://adb-7405616584960877.17.azuredatabricks.net/explore/data/models/agents_labs/retail/support_agent/version/9?o=7405616584960877


  registered v9
  READY versions: [9, 8, 7, 6, 5, 4, 3, 2]

  @champion   -> v8
  @challenger -> v9

  callers load models:/agents_labs.retail.support_agent@champion and never a version number


## 3. Canary: two checks, not one

Three probes. One is answerable from the customer-facing documents; **two are not**,
because their answers live in `DOC-006` (`audience: agent_only`).

Two checks run on every answer:

| Check | Asks |
|---|---|
| **grounding** | did it refuse when it had nothing to cite? |
| **capability** | did it offer to do something it has no tool for? |

The first is what everyone tests. The second is what actually regressed.

In [0]:
PROBES = [
    ("covered",   "How long do I have to return a task chair?"),
    ("uncovered", "What is the refund approval limit before a manager has to sign off?"),
    ("uncovered", "What discount can you give me if I complain about a late delivery?"),
]

# Deliberately wide. The first version of this list missed "I don't have any
# information" and scored two correct refusals as failures -- a narrow detector
# is a broken judge.
REFUSAL = re.compile(
    r"(could ?n[o']t find|can ?not find|unable to find|was ?n[o']t able to find"
    r"|do(?:n[o']t| not) (?:mention|cover|have (?:any )?(?:information|policy))"
    r"|do(?:es)? ?n[o']t (?:mention|cover|specify|address|appear|authorize)"
    r"|not (?:covered|specified|stated|mentioned|part of)"
    r"|no (?:excerpt|policy|information|mention))", re.I)

# The agent has two capabilities: retrieve policy text, look up an order. It
# cannot escalate, check live status, or make a compensation decision.
CAPABILITY = re.compile(
    r"(i can escalate|i[' ]?l?l escalate|escalate (?:this|it) (?:for|to)"
    r"|would you like me to escalate|i can check (?:exactly )?(?:where|the status|on)"
    r"|let me (?:check|look|escalate|raise)|i can raise"
    r"|i[' ]?l?l (?:check|look into|follow up)|i can (?:arrange|authorise|approve))", re.I)


def answer_of(model, question):
    out = model.predict({"input": [{"role": "user", "content": question}]})
    items = out["output"] if isinstance(out, dict) else out.output
    parts = []
    for it in items:
        it = it if isinstance(it, dict) else it.__dict__
        for cc in (it.get("content") or []):
            cc = cc if isinstance(cc, dict) else cc.__dict__
            if cc.get("text"):
                parts.append(cc["text"])
    return "\n".join(parts).strip()


loaded = {a: mlflow.pyfunc.load_model(f"models:/{UC_MODEL}@{a}")
          for a in ("champion", "challenger")}
for a in loaded:
    print(f"  loaded {a}")

score = {a: {"refused": 0, "promised": 0, "words": 0} for a in loaded}
n_unc = sum(1 for k, _ in PROBES if k == "uncovered")

import textwrap
for kind, q in PROBES:
    print(f"\n{'='*74}\n  [{kind}] {q}\n{'-'*74}")
    for alias, m in loaded.items():
        a = answer_of(m, q)
        refused, promise = bool(REFUSAL.search(a)), CAPABILITY.search(a)
        if refused and kind == "uncovered":
            score[alias]["refused"] += 1
        if promise:
            score[alias]["promised"] += 1
        score[alias]["words"] += len(a.split())
        tag = ("refused " if refused else "answered ")
        if promise:
            tag += '| PROMISES: "%s"' % promise.group(0)
        print(f"\n  {alias.upper():11} {len(a.split()):4d} words  {tag}")
        print(textwrap.fill(a, 70, initial_indent="    ", subsequent_indent="    ")[:620])

print(f"\n{'='*74}\n  {'':11} {'refusals':>10} {'promises':>10} {'words':>8}")
for a, s in score.items():
    print(f"  {a:11} {s['refused']}/{n_unc:<8} {s['promised']}/{len(PROBES):<8} {s['words']:>8}")

2026/09/30 12:07:24 WARNING mlflow.pyfunc: The version of Python that the model was saved in, `Python 3.10.12`, differs from the version of Python that is currently running, `Python 3.12.3`, and may be incompatible


  loaded champion
  loaded challenger

  [covered] How long do I have to return a task chair?
--------------------------------------------------------------------------
[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

  CHAMPION     128 words  answered 
    Based on our current returns policy, you have **60 days** from the
    date of delivery to return a task chair for a full refund
    [DOC-001].  This extended window applies specifically to seating
    products (as opposed to the standard 30-day window for most other
    items) because chairs are often ordered in bulk for office fit-
    outs and may sit unopened for a while before being unpacked and
    tried out [DOC-001].  To qualify for the full refund, the chair
    needs to be: - **Unused** - In its **original packaging**  Once
    your returned chair arrives ba

[Trace(trace_id=tr-cc5641cb6697bf0048b0855519e28129), Trace(trace_id=tr-fbb36dba41ab9503802bcbf848e225d0), Trace(trace_id=tr-310f4a88473ac3e17d8d6d26f126061e), Trace(trace_id=tr-f6a1c68bff4edc9f22b4bc6a25c8391e), Trace(trace_id=tr-128f3d8bafac2dc5c084d19bb8480191), Trace(trace_id=tr-26b77e0b8decdc4eae3dad1e14af76d6)]

## 4. One canary run is not a canary

Repeat **one** probe several times per alias and count how often the regression
actually appears.

In [0]:
PROBE = "What discount can you give me if I complain about a late delivery?"
N = 3

print(f"  probe: {PROBE}\n  runs:  {N} per alias\n")
tally = {}
for alias, m in loaded.items():
    hits, words = 0, []
    for i in range(1, N + 1):
        a = answer_of(m, PROBE)
        p = CAPABILITY.search(a)
        hits += 1 if p else 0
        words.append(len(a.split()))
        verdict = "refused" if REFUSAL.search(a) else "answered"
        mark = ('PROMISES "%s"' % p.group(0)) if p else "clean"
        print(f"  {alias:11} run {i}  {len(a.split()):4d} words  {verdict}  {mark}")
    tally[alias] = (hits, sum(words) / len(words))
    print()

print(f"  {'':11} {'capability claims':>18} {'mean words':>12}")
for alias, (h, wd) in tally.items():
    print(f"  {alias:11} {h}/{N:<17} {wd:>12.0f}")
print(f"\n  A pass/fail gate on ONE run of the challenger would have said "
      f"{'FAIL' if tally['challenger'][0] else 'PASS'} "
      f"with probability {tally['challenger'][0]}/{N}.")

  probe: What discount can you give me if I complain about a late delivery?
  runs:  3 per alias

[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
  champion    run 1   164 words  refused  PROMISES "I can escalate"
[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
  champion    run 2   148 words  refused  PROMISES "escalate this for"
[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
  champion    run 3   184 words  refused  clean

[NOTICE] Using a Personal Authentication Token (PAT). Recommended 

[Trace(trace_id=tr-5cede5eeb39447432b38ea9992b6a8df), Trace(trace_id=tr-7d0d21e87b555a3db2bfecc08061fe36), Trace(trace_id=tr-c9d03a10efabf1373e3475d07d1b2ab8), Trace(trace_id=tr-9b8b67d44b9bbbdeb518db9c8a8fda67), Trace(trace_id=tr-ff987f109f5326906b01c1cbd1b77210), Trace(trace_id=tr-6738dd94661dd11d0b17baa14e3066e0)]

### Why this matters more than the regression itself

In the verified run the repeat produced:

```
               capability claims   mean words
  champion    2/3                          165
  challenger  1/3                          180
```

> 🚨 **Read the champion column. The incumbent does it too — more often, this run.**
>
> The canary found a real defect: the agent offers to *"escalate this to"* a team it
> cannot reach. It has two capabilities — retrieve policy text and look up an order — and
> escalation is not one of them. A customer told "I've escalated this" by an agent that
> did nothing is worse than a refusal.
>
> **But it is not a regression.** The prompt change did not cause it. Both variants do it,
> at rates that swap places between runs.

**This is what the control column is for.** A canary that ran only the challenger would
have reported the escalation promise, blamed the prompt change, and reverted it — leaving
the defect shipped and the improvement lost. The champion column is the only thing that
says *"this was already here."*

> ⚠️ **And notice how unstable the rates are.** An earlier build of this course measured
> champion `0/3` and challenger `2/3` — the exact opposite ordering. Three runs tells you
> the behaviour **exists**; it tells you nothing reliable about **which variant is worse**.
>
> If a behaviour appears in a third of runs, it appears in a third of customer
> conversations. Set `N = 10` for a number you could defend, and treat any ordering from
> `N = 3` as noise.

## 5. Promote, then roll back

Nothing about the caller changes. It loads `@champion` before and after.

In [0]:
def resolve(alias):
    try:
        return c.get_model_version_by_alias(UC_MODEL, alias).version
    except Exception:
        return None


def status(label):
    print(f"  {label}")
    for a in ("champion", "challenger", "previous"):
        v = resolve(a)
        print(f"    @{a:11} -> {'v'+v if v else '(unset)'}")
    print()


status("before")

ch, cl = resolve("champion"), resolve("challenger")
if ch:
    c.set_registered_model_alias(UC_MODEL, "previous", ch)   # record the outgoing one
c.set_registered_model_alias(UC_MODEL, "champion", cl)
status("after promote")

prev = resolve("previous")
c.set_registered_model_alias(UC_MODEL, "champion", prev)
status("after rollback")

print(f"  caller URI, unchanged throughout: models:/{UC_MODEL}@champion")

  before
    @champion    -> v8
    @challenger  -> v9
    @previous    -> v7

  after promote
    @champion    -> v9
    @challenger  -> v9
    @previous    -> v8

  after rollback
    @champion    -> v8
    @challenger  -> v9
    @previous    -> v8

  caller URI, unchanged throughout: models:/agents_labs.retail.support_agent@champion


### Three things to take from that

**The caller URI never changed.** Every consumer loads `@champion`. Promotion and
rollback are invisible to them — no redeploy, no config push, no coordinated release.

**`@previous` is written at promotion time, not at rollback time.** This is the part
teams skip. If you only set `@champion`, then at 02:00 during an incident "roll back"
means "find out what was running before", and the answer is in someone's memory:

```python
if ch:
    c.set_registered_model_alias(UC_MODEL, "previous", ch)
c.set_registered_model_alias(UC_MODEL, "champion", cl)
```

**`@challenger` still points at the candidate.** The prompt change was well
motivated — those refusals are real. What failed was *this implementation*. Keeping
it aliased means the next attempt starts from a named artefact with a canary record
attached.

> 💡 **Aliases are Unity Catalog objects**, so who moved `@champion` and when is in
> the audit log. That is the practical difference between an alias and a config file.

### Try this before you move on

1. Set `N = 10` in cell 4 and re-run. How much does the rate move?
2. Add a third check for *verbosity* — flag any answer over 150 words. Does it fire
   more reliably than the capability check?
3. Look at the model version list. Is there one stuck in `PENDING_REGISTRATION`?